In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 15 — ALÉM DA ACURÁCIA: A MATRIZ DE CONFUSÃO E SUAS MÉTRICAS

> "Acurácia é a manchete. A Matriz de Confusão é a reportagem investigativa. A manchete diz quem ganhou; a reportagem conta como o jogo foi jogado, quem falhou nos momentos cruciais e por quê."
> — Um Analista de Dados Calejado

97,71%. O número flutuava na tela, troféu do meu trabalho — a maior acurácia que eu alcançara. Por um momento, senti o impulso de comemorar. Mas a lembrança de Helena era uma âncora. O OncoClassify 1.0 também tinha acurácia alta. E, ainda assim, falhou com ela — de uma forma que a acurácia sozinha não conseguia descrever.

Eu precisava de um microscópio, não de um telescópio. Precisava desmontar a acurácia em seus componentes e olhar cada tipo de erro que o modelo cometia. Precisava da **Matriz de Confusão**, agora aplicada ao meu melhor modelo.

## A Matriz de Confusão do Modelo Afinado

Vamos avaliar o *pipeline* otimizado no Capítulo 14 (`k=25, C=10, gamma=0.01`). Para uma estimativa honesta, usamos `cross_val_predict`: cada previsão vem de um modelo que **não** viu aquele ponto no treino. Assim, a matriz reflete o desempenho de generalização, não a memorização.

Lembrando as métricas (classe positiva = Maligno):
**Recall** = VP / (VP + FN) — dos malignos reais, quantos foram encontrados.
**Precisão** = VP / (VP + FP) — das previsões "Maligno", quantas acertaram.
**F1** = média harmônica de ambas.

#### Código 15.1: Matriz de Confusão e Relatório de Classificação


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
from oncoclassify_utils import X_train, y_train

def info_mutua(X, y):
    return mutual_info_classif(X, y, random_state=42)

modelo = Pipeline([
    ("selecao", SelectKBest(info_mutua, k=25)),
    ("escala",  StandardScaler()),
    ("svm",     SVC(kernel="rbf", C=10, gamma=0.01, random_state=42)),
])

# previsoes fora-da-amostra, via validacao cruzada
y_pred = cross_val_predict(modelo, X_train, y_train, cv=5)

print(classification_report(y_train, y_pred,
      target_names=["Maligno (0)", "Benigno (1)"], digits=3))
print("Matriz de confusao [linhas=real, colunas=previsto]:")
print(confusion_matrix(y_train, y_pred, labels=[0, 1]))

              precision    recall  f1-score   support

 Maligno (0)      0.988     0.949     0.968       176
 Benigno (1)      0.971     0.993     0.982       304

    accuracy                          0.977       480
   macro avg      0.980     0.971     0.975       480
weighted avg      0.977     0.977     0.977       480

Matriz de confusao [linhas=real, colunas=previsto]:
[[167   9]
 [  2 302]]


![Matriz de confusão do modelo afinado](media/figura_15_1.png)

Agora temos a anatomia exata do desempenho:

**VP = 167** malignos corretamente identificados; **VN = 302** benignos corretos.

**FP = 2** — apenas dois alarmes falsos. A **precisão da classe maligna é 98,8%**: quando o modelo diz "Maligno", quase sempre acerta.

**FN = 9** (célula destacada em vermelho) — nove tumores malignos classificados como benignos. É o número que importa. O **recall maligno é 94,9%**: o modelo encontra 167 dos 176 cânceres do treino, mas deixa **nove** passarem.

A acurácia de 97,7% escondia esses nove. Cada um é uma Helena em potencial. É esse número que vamos atacar no próximo capítulo.

> **📌 CHECKLIST DO CAPÍTULO 15**
>
> ✅ Entende por que a acurácia engana e por que a Matriz de Confusão é a primeira análise a fazer.
>
> ✅ Sabe definir VP, VN, FP, FN e derivar Precisão, Recall e F1.
>
> ✅ Usou `cross_val_predict` para uma avaliação **fora-da-amostra**, sem *leakage*.
>
> ✅ Executou o Código 15.1 e identificou **9 Falsos Negativos** como a principal preocupação (recall maligno 94,9%).
>
> **⚠️ CRÍTICO:** Em qualquer classificação, comece pela Matriz de Confusão. Ela dá o contexto para entender o que a acurácia realmente significa.

A matriz foi um balde de água fria necessário. Ela substituiu a falsa segurança dos 97,7% pela dura realidade: **nove**. Nove cânceres não detectados. O número ecoava.

Eu não poderia eliminar todo erro, mas talvez pudesse influenciar o *tipo* de erro. O modelo não decide só "maligno" ou "benigno" — nos bastidores, ele calcula um escore de confiança e o compara a um limiar. E se eu tornasse esse limiar mais cauteloso, capturando alguns daqueles nove, mesmo ao custo de mais alarmes falsos? Eu precisava assumir o controle do **threshold de decisão**.
